<a href="https://colab.research.google.com/github/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/blob/main/notebooks/02_entrenamiento_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Entrenamiento de modelos YOLO para detección de humo y fuego

Baseline del proyecto: YOLOv8n sobre D-Fire.

## Estado

El baseline **ya está entrenado**: 30 épocas en Colab, y la validación se corrió
después sobre una Tesla T4. Sus resultados están versionados en
`reports/results/yolov8n_baseline/`, así que este notebook no vuelve a entrenar
(`RUN_TRAINING = False`).

## Cómo correrlo

- **Solo para regenerar `metrics_summary.csv`**, que es lo único que el notebook
  05 necesita de este experimento: alcanza con las celdas de setup, hasta *Carga
  de configuración del experimento*, y la de *Reconstrucción del resumen*, al
  final. No hace falta GPU, ni Drive, ni el dataset.
- **De punta a punta**: en Colab con GPU. Las celdas de dataset bajan D-Fire con
  `kagglehub` y lo validan, y las de pesos leen la corrida desde
  `MyDrive/VCII_DFire/runs/yolov8n_baseline/`.

## Salidas esperadas

El entrenamiento genera:

- pesos del modelo (`best.pt` y `last.pt`);
- métricas de entrenamiento y validación;
- curvas de desempeño;
- matriz de confusión;
- carpeta de resultados asociada al experimento.


In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)
print("Directorio actual:", Path.cwd())


In [ ]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
# Rama del repositorio desde la que se clona y a la que se commitean los
# resultados. Mientras el PR este abierto tiene que apuntar a la rama del PR;
# una vez mergeado, cambiar a "main".
REPO_BRANCH = "feat/modelos-adicionales-deteccion"

RAW_REQUIREMENTS = (
    "https://raw.githubusercontent.com/Gabriela-Sol/"
    f"{REPO_NAME}/{REPO_BRANCH}/requirements.txt"
)

if IN_COLAB:
    !pip install -q -r {RAW_REQUIREMENTS}

print("Dependencias instaladas.")


In [ ]:
# ============================================================
# Verificación de GPU
# ============================================================
# Solo hace falta para entrenar o para volver a correr `model.val()`. La
# reconstrucción del resumen, al final del notebook, no usa la GPU.

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE_NAME = torch.cuda.get_device_name(0)
    print("GPU:", DEVICE_NAME)
else:
    DEVICE_NAME = "cpu"
    print("No se detectó GPU. Para entrenar, activarla en Colab:")
    print("Entorno de ejecución > Cambiar tipo de entorno > GPU.")


In [ ]:
# ============================================================
# Almacenamiento de trabajo (Drive en Colab)
# ============================================================
# WORK_DIR es la raíz escribible del entorno: de ahí salen el clon del repo y el
# YAML del dataset.

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
    DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
    print("Carpeta de corridas:", DRIVE_RUNS_DIR)

    WORK_DIR = Path("/content")
else:
    WORK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("WORK_DIR:", WORK_DIR)


In [ ]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

if IN_COLAB:
    PROJECT_DIR = WORK_DIR / REPO_NAME

    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git checkout {REPO_BRANCH}
        !git pull origin {REPO_BRANCH}
    else:
        print("Clonando repositorio...")
        %cd {WORK_DIR}
        !git clone -b {REPO_BRANCH} {REPO_URL}.git
        %cd {PROJECT_DIR}
else:
    # Ya estamos dentro del repo: no hay nada que clonar.
    PROJECT_DIR = WORK_DIR

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:", os.listdir(PROJECT_DIR))


In [ ]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "yolov8n_baseline.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración: {EXPERIMENT_CONFIG_PATH}"
    )

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
family = experiment_config["experiment"]["family"]

# `output.project` del YAML apunta a Drive, que solo existe en Colab. La ruta
# efectiva se resuelve acá en vez de editar `experiment_config`, para que el
# experiment_config_used.yaml salga igual en los dos entornos y los diffs entre
# corridas no muestren cambios de ruta que no son hiperparámetros.
if IN_COLAB:
    RUNS_DIR = Path(experiment_config["output"]["project"])
else:
    RUNS_DIR = WORK_DIR / "runs"

print("Experimento:", experiment_name)
print("Familia:", family)
print("Modelo:", model_name)
print("Descripción:", experiment_config["experiment"]["description"])
print("Corridas en:", RUNS_DIR)


In [ ]:
# ============================================================
# Descarga o localización del dataset
# ============================================================
# Solo hace falta para entrenar o para volver a validar. La reconstrucción del
# resumen, al final del notebook, no toca el dataset.

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"

dataset_root = Path(kagglehub.dataset_download(DATASET_ID))

print("Dataset descargado/localizado en:")
print(dataset_root)

print("\nContenido del directorio raíz del dataset:")
for item in dataset_root.iterdir():
    print("-", item)


In [9]:
# ============================================================
# Detección estructura del dataset
# ============================================================

def find_yolo_dataset_dir(root: Path) -> Path:
    """
    Busca automáticamente la carpeta que contiene la estructura esperada:
    train/images, train/labels, val/images, val/labels.
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        train_images = candidate / "train" / "images"
        train_labels = candidate / "train" / "labels"
        val_images = candidate / "val" / "images"
        val_labels = candidate / "val" / "labels"

        if (
            train_images.exists()
            and train_labels.exists()
            and val_images.exists()
            and val_labels.exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró una estructura YOLO válida con train/images, train/labels, val/images y val/labels."
    )

DATA_DIR = find_yolo_dataset_dir(dataset_root)

print("Carpeta de datos YOLO detectada:")
print(DATA_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"{split}: existe={split_dir.exists()} -> {split_dir}")

Carpeta de datos YOLO detectada:
/kaggle/input/smoke-fire-detection-yolo/data
train: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/train
val: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/val
test: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/test


In [ ]:
# ============================================================
# Creación del archivo YAML para YOLO
# ============================================================

DFIRE_YAML = WORK_DIR / "dfire_dataset.yaml"

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dfire_config, file, sort_keys=False, allow_unicode=True)

print("Archivo YAML creado en:", DFIRE_YAML)
print()
print(DFIRE_YAML.read_text())


In [11]:
# # ============================================================
# validación rápida del dataset
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_split_files(data_dir: Path, split:str):
  images_dir = data_dir / split / "images"
  labels_dir = data_dir / split / "labels"

  image_files = [f for f in images_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
  label_files = [f for f in labels_dir.iterdir() if f.suffix.lower() == ".txt"]

  return len(image_files), len(label_files)

for split in ["train", "val", "test"]:
  images_count, labels_count = count_split_files(DATA_DIR, split)
  print(f"{split:5s} | imágenes: {images_count:6d} | etiquetas: {labels_count:6d}")




train | imágenes:  14122 | etiquetas:  14122
val   | imágenes:   3099 | etiquetas:   3099
test  | imágenes:   4306 | etiquetas:   4306


In [12]:
# ============================================================
# Validación básica de etiquetas YOLO
# ============================================================

def validate_yolo_labels(labels_dir: Path):
    """
    Valida etiquetas YOLO:
    class_id x_center y_center width height
    con coordenadas normalizadas entre 0 y 1.
    """
    invalid_entries = []
    empty_files = 0
    total_boxes = 0

    for label_path in labels_dir.glob("*.txt"):
        content = label_path.read_text().strip()

        if content == "":
            empty_files += 1
            continue

        for line_number, line in enumerate(content.splitlines(), start=1):
            parts = line.split()

            if len(parts) != 5:
                invalid_entries.append((str(label_path), line_number, "Cantidad de campos != 5", line))
                continue

            try:
                class_id = int(float(parts[0]))
                coords = list(map(float, parts[1:]))
            except ValueError:
                invalid_entries.append((str(label_path), line_number, "Valores no numéricos", line))
                continue

            if class_id not in [0, 1]:
                invalid_entries.append((str(label_path), line_number, "Clase fuera de rango", line))
                continue

            if not all(0 <= value <= 1 for value in coords):
                invalid_entries.append((str(label_path), line_number, "Coordenadas fuera de [0, 1]", line))
                continue

            total_boxes += 1

    return {
        "empty_files": empty_files,
        "total_boxes": total_boxes,
        "invalid_entries": invalid_entries,
    }

for split in ["train", "val", "test"]:
    labels_dir = DATA_DIR / split / "labels"
    validation = validate_yolo_labels(labels_dir)

    print(f"\nSplit: {split}")
    print("Labels vacíos:", validation["empty_files"])
    print("Bounding boxes válidos:", validation["total_boxes"])
    print("Entradas problemáticas:", len(validation["invalid_entries"]))

    if validation["invalid_entries"][:5]:
        print("Primeras entradas problemáticas:")
        for entry in validation["invalid_entries"][:5]:
            print(entry)


Split: train
Labels vacíos: 6458
Bounding boxes válidos: 17432
Entradas problemáticas: 0

Split: val
Labels vacíos: 1375
Bounding boxes válidos: 3932
Entradas problemáticas: 0

Split: test
Labels vacíos: 2005
Bounding boxes válidos: 5185
Entradas problemáticas: 8
Primeras entradas problemáticas:
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11606.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.2578125 0.4986111111111111 0.50625 1.0027777777777778')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11600.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.496875 0.4222222222222222 1.0562500000000001 0.8388888888888889')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11090.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.5437500000000001 0.5013888888888889 0.5375 1.0027777777777778')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB10821.txt', 3, 'Coordenadas fuera de [0, 1]', '0 0.49843750000000003 0.3388888888888889 1.0093750000000001 0.616666

In [ ]:
# ============================================================
# Función de entrenamiento desde configuración
# ============================================================

from ultralytics import YOLO
import pandas as pd

def train_from_config(config: dict, data_yaml: Path, project_dir: Path) -> dict:
    """
    Entrena un modelo YOLO a partir de un archivo de configuración YAML.
    Los resultados se guardan en una carpeta específica por experimento.

    `project_dir` se recibe en vez de leerse de `config["output"]["project"]`
    porque esa ruta del YAML apunta a Drive: quien llama ya resolvió la carpeta
    que corresponde al entorno.
    """

    exp = config["experiment"]
    train_cfg = config["training"]

    experiment_name = exp["name"]
    model_name = exp["model"]

    project_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Experimento: {experiment_name}")
    print(f"Familia: {exp['family']}")
    print(f"Modelo: {model_name}")
    print(f"Resultados en: {project_dir / experiment_name}")
    print("=" * 80)

    start_time = time.time()

    model = YOLO(model_name)

    results = model.train(
        data=str(data_yaml),
        epochs=train_cfg["epochs"],
        imgsz=train_cfg["imgsz"],
        batch=train_cfg["batch"],
        patience=train_cfg["patience"],
        optimizer=train_cfg["optimizer"],
        lr0=train_cfg["lr0"],
        seed=train_cfg["seed"],
        project=str(project_dir),
        name=experiment_name,
        exist_ok=True,
        plots=True,
    )

    elapsed_time = time.time() - start_time

    experiment_dir = project_dir / experiment_name
    best_model_path = experiment_dir / "weights" / "best.pt"
    last_model_path = experiment_dir / "weights" / "last.pt"
    results_csv = experiment_dir / "results.csv"

    summary = {
        "experiment": experiment_name,
        "family": exp["family"],
        "model": model_name,
        "epochs": train_cfg["epochs"],
        "imgsz": train_cfg["imgsz"],
        "batch": train_cfg["batch"],
        "training_time_min": round(elapsed_time / 60, 2),
        "experiment_dir": str(experiment_dir),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "results_csv": str(results_csv),
        "best_exists": best_model_path.exists(),
        "last_exists": last_model_path.exists(),
    }

    print("\nEntrenamiento finalizado.")
    print("Tiempo total [min]:", summary["training_time_min"])
    print("Best model:", best_model_path)
    print("Last model:", last_model_path)

    return summary


In [ ]:
# ============================================================
# Ejecución del experimento baseline
# ============================================================
# El baseline ya está entrenado: sus 30 épocas corrieron en Colab y sus
# resultados están versionados en reports/results/yolov8n_baseline/. Esta celda
# queda como registro de cómo se lanzó, pero no vuelve a entrenar.

RUN_TRAINING = False

if RUN_TRAINING:
    experiment_summary = train_from_config(
        config=experiment_config,
        data_yaml=DFIRE_YAML,
        project_dir=RUNS_DIR,
    )

    summary_df = pd.DataFrame([experiment_summary])
    display(summary_df)
else:
    print("RUN_TRAINING = False: no se entrena.")
    print("Los resultados de la corrida original están en el repositorio, en")
    print(f"  reports/results/{experiment_name}/")


In [ ]:
# ============================================================
# Verificación de pesos guardados
# ============================================================
# Los pesos no están en el repositorio, pesan de más: viven en Drive, en
# VCII_DFire/runs/yolov8n_baseline/weights/. Solo hacen falta para reentrenar o
# para volver a validar; la reconstrucción del resumen no los usa.

experiment_dir = RUNS_DIR / experiment_name

best_model_path = experiment_dir / "weights" / "best.pt"
last_model_path = experiment_dir / "weights" / "last.pt"

print("Carpeta del experimento:", experiment_dir)
print("Best model existe:", best_model_path.exists(), best_model_path)
print("Last model existe:", last_model_path.exists(), last_model_path)


In [ ]:
# ============================================================
# Guardar resumen general de experimentos
# ============================================================
# Bookkeeping de las corridas, en Drive. Solo tiene sentido si se entrenó, así
# que SUMMARY_PATH se resuelve adentro del if: DRIVE_PROJECT_DIR no existe fuera
# de Colab y referenciarlo antes rompía la corrida local con NameError.

if RUN_TRAINING:
    SUMMARY_PATH = DRIVE_PROJECT_DIR / "experiment_summary.csv"

    if SUMMARY_PATH.exists():
        previous_summary = pd.read_csv(SUMMARY_PATH)
        updated_summary = pd.concat(
            [previous_summary, summary_df],
            ignore_index=True
        )
    else:
        updated_summary = summary_df.copy()

    updated_summary.to_csv(SUMMARY_PATH, index=False)
    display(updated_summary)

    print("Resumen actualizado en:", SUMMARY_PATH)
else:
    print("No se actualizó el resumen porque no se entrenó.")


In [ ]:
# ============================================================
# Copiar resultados principales al repositorio local
# ============================================================
# Copia los artefactos que Ultralytics dejó en la carpeta de la corrida. Con
# RUN_TRAINING = False y sin Drive montado no hay nada que copiar: los archivos
# de la corrida original ya están versionados y la celda solo informa.

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name
REPORTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

for filename in files_to_copy:
    src = experiment_dir / filename
    dst = REPORTS_RESULTS_DIR / filename

    if src.exists():
        shutil.copy(src, dst)
        print("Copiado:", dst)
    else:
        print("No encontrado:", src)

# Guardar también la configuración usada para este experimento
used_config_path = REPORTS_RESULTS_DIR / "experiment_config_used.yaml"

with open(used_config_path, "w", encoding="utf-8") as file:
    yaml.safe_dump(experiment_config, file, sort_keys=False, allow_unicode=True)

print("Config usada guardada en:", used_config_path)


## Reconstrucción del resumen para la comparación

La celda que sigue arma `reports/results/yolov8n_baseline/metrics_summary.csv`,
que es lo único que el notebook 05 necesita de este experimento.

No entrena ni valida: reusa los números de la validación que ya se corrió sobre
los pesos entrenados, en una Tesla T4. Corre en cualquier máquina, sin GPU, sin
Drive y sin el dataset.


In [ ]:
# ============================================================
# Reconstrucción del metrics_summary.csv del baseline
# ============================================================
# Esta celda permite que YOLOv8n participe de la comparación del notebook 05.
#
# No entrena ni valida: reusa la validación que ya se corrió sobre los pesos
# entrenados, en una Tesla T4, y que dejó sus curvas en la carpeta
# VCII_DFire/runs/yolov8n_baseline_val de Drive. Volver a correr model.val()
# daría los mismos números, pero exigiría GPU, el dataset y los pesos; medirlo
# en otra máquina cambiaría `fps` y lo volvería incomparable con las filas de
# Faster R-CNN y RT-DETR, que se midieron en esa misma T4.
#
# Salida original de esa corrida, tal como quedó en los outputs del notebook en
# el commit 735a75c (la celda que escribía `name="yolov8n_baseline_val"`):
#
#     Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients
#                    Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
#                      all       3094       3917      0.748      0.683      0.752      0.431
#                    smoke       1545       1751      0.801      0.742      0.808      0.495
#                     fire        875       2166      0.695      0.624      0.696      0.367
#     Speed: 1.0ms preprocess, 2.9ms inference, 0.0ms loss, 1.5ms postprocess per image
#
# La tabla imprime a tres decimales. mAP50 y mAP50-95 van con los floats
# completos, que la misma corrida imprimió aparte vía metrics.box.map50 y .map.
#
# Las 3094 imágenes son cinco menos que las 3099 del split: esas cinco las
# saltearon los permisos de solo lectura del filesystem donde estaba el dataset,
# no un problema de las imágenes.

import pandas as pd

from src.reporting.summary import write_metrics_summary

# --- lo que imprimió Ultralytics ----------------------------------------
PARAMETROS = 3_006_038
PRECISION, RECALL = 0.748, 0.683
MAP50, MAP50_95 = 0.752349089580163, 0.43131541206288115
MAP50_SMOKE, MAP50_FIRE = 0.808, 0.696
MAP50_95_SMOKE, MAP50_95_FIRE = 0.495, 0.367
MS_INFERENCE, MS_POSTPROCESS = 2.9, 1.5
DEVICE_MEDICION = "Tesla T4"

# --- derivados ----------------------------------------------------------
# Se recalculan con las mismas fórmulas que build_ultralytics_metrics_row, la
# función que arma esta fila en los notebooks 03 y 04, en vez de transcribirse:
# así f1 y fps no pueden quedar desfasados de los números de arriba.
f1 = 2 * PRECISION * RECALL / (PRECISION + RECALL)
fps = 1000.0 / (MS_INFERENCE + MS_POSTPROCESS)

# El tiempo de entrenamiento sale del results.csv versionado, en segundos.
REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name
train_time_min = pd.read_csv(REPORTS_RESULTS_DIR / "results.csv")["time"].iloc[-1] / 60

training_cfg = experiment_config["training"]

resumen = {
    "experiment": experiment_name,
    "family": family,
    "model": model_name,
    "params_M": round(PARAMETROS / 1e6, 2),
    "epochs": training_cfg["epochs"],
    "imgsz": training_cfg["imgsz"],
    "batch": training_cfg["batch"],
    "train_time_min": round(train_time_min, 2),
    "mAP50": round(MAP50, 4),
    "mAP50_95": round(MAP50_95, 4),
    "precision": round(PRECISION, 4),
    "recall": round(RECALL, 4),
    "f1": round(f1, 4),
    "mAP50_smoke": round(MAP50_SMOKE, 4),
    "mAP50_fire": round(MAP50_FIRE, 4),
    "mAP50_95_smoke": round(MAP50_95_SMOKE, 4),
    "mAP50_95_fire": round(MAP50_95_FIRE, 4),
    "fps": round(fps, 2),
    "device": DEVICE_MEDICION,
    "split": "val",
}

# write_metrics_summary valida el esquema antes de tocar el disco: si faltara o
# sobrara una columna corta acá, en vez de dejar un CSV que el 05 no puede leer.
destino = REPORTS_RESULTS_DIR / "metrics_summary.csv"
display(write_metrics_summary(destino, resumen).T.rename(columns={0: "valor"}))
print("Guardado en:", destino)

# Las curvas de esa validación quedaron en Drive y no en el repositorio. Bajarlas
# deja al baseline con los mismos archivos que Faster R-CNN; el notebook 05 no
# las necesita, solo lee el CSV.
faltantes = [
    nombre
    for nombre in ["PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]
    if not (REPORTS_RESULTS_DIR / nombre).exists()
]

if faltantes:
    print("\nFaltan las curvas:", ", ".join(faltantes))
    print("Bajarlas de MyDrive/VCII_DFire/runs/yolov8n_baseline_val/ a")
    print(" ", REPORTS_RESULTS_DIR)


In [ ]:
# ============================================================
# Publicación del resumen del baseline
# ============================================================

import subprocess

%cd {PROJECT_DIR}

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

pull_result = subprocess.run(
    ["git", "pull", "--rebase", "origin", REPO_BRANCH], text=True, capture_output=True
)
print(pull_result.stdout, pull_result.stderr)

if pull_result.returncode != 0:
    raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

for path in [f"reports/results/{experiment_name}/", "configs/experiments/yolov8n_baseline.yaml"]:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)

status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
print(status.stdout)

if not status.stdout.strip():
    print("No hay cambios nuevos para commitear.")
else:
    subprocess.run(
        ["git", "commit", "-m", f"results: metrics_summary de {experiment_name}"],
        check=True,
    )
    print(f"Commit creado. Para publicarlo: !git push origin {REPO_BRANCH}")
